<a href="https://colab.research.google.com/github/joaopedrodias07/wave_height_forecast/blob/master/leitura_variaveis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
library("ncdf4")
library("tidyverse")
library("lubridate")

In [ ]:
inspecionar_nc <- function(arquivo) {
  cat("\n", paste(rep("=", 50), collapse=""), "\n")
  cat("ARQUIVO:", arquivo, "\n")

  nc <- nc_open(arquivo)

  cat("\nVARIÁVEIS:\n")
  for (var in names(nc$var)) {
    v <- nc$var[[var]]
    cat(sprintf("  %s | %s | unidade: %s\n",
                var, v$longname, v$units))
  }

  cat("\nDIMENSÕES:\n")
  for (dim in names(nc$dim)) {
    d <- nc$dim[[dim]]
    cat(sprintf("  %s | tamanho: %d\n", dim, d$len))
  }

  nc_close(nc)
}

In [ ]:
arquivos <- list.files(pattern = "\\.nc$")
cat("Arquivos encontrados:", length(arquivos), "\n")
for (arq in arquivos) inspecionar_nc(arq)

Arquivos encontrados: 9 

ARQUIVO: variavel1.nc 

VARIÁVEIS:
  number | ensemble member numerical id | unidade: 1
  expver | expver | unidade: 
  u10 | 10 metre U wind component | unidade: m s**-1

DIMENSÕES:
  valid_time | tamanho: 8760
  latitude | tamanho: 1
  longitude | tamanho: 2

ARQUIVO: variavel2.nc 

VARIÁVEIS:
  number | ensemble member numerical id | unidade: 1
  expver | expver | unidade: 
  v10 | 10 metre V wind component | unidade: m s**-1

DIMENSÕES:
  valid_time | tamanho: 8760
  latitude | tamanho: 1
  longitude | tamanho: 2

ARQUIVO: variavel3.nc 

VARIÁVEIS:
  number | ensemble member numerical id | unidade: 1
  expver | expver | unidade: 
  pp1d | Peak wave period | unidade: s

DIMENSÕES:
  valid_time | tamanho: 8760
  latitude | tamanho: 1
  longitude | tamanho: 1

ARQUIVO: variavel4.nc 

VARIÁVEIS:
  number | ensemble member numerical id | unidade: 1
  expver | expver | unidade: 
  msl | Mean sea level pressure | unidade: Pa

DIMENSÕES:
  valid_time | tamanho: 87

In [ ]:
nc <- nc_open("variavel1.nc")
lat <- ncvar_get(nc, "latitude")
lon <- ncvar_get(nc, "longitude")
nc_close(nc)

cat("Latitudes:", lat, "\n")
cat("Longitudes:", lon, "\n")

# Santos: -23.93, -46.33
santos_lat <- -23.93
santos_lon <- -46.33

# Distância só na longitude pois latitude tem só 1 ponto
dist_lon <- abs(lon - santos_lon)
idx_lon  <- which.min(dist_lon)

cat("\nDistâncias para Santos:\n")
cat(sprintf("  lon=%.2f → distância=%.4f\n", lon[1], dist_lon[1]))
cat(sprintf("  lon=%.2f → distância=%.4f\n", lon[2], dist_lon[2]))
cat(sprintf("\nPonto mais próximo: longitude %.2f (índice %d)\n",
            lon[idx_lon], idx_lon))

Latitudes: -24 
Longitudes: -46.5 -46.25 

Distâncias para Santos:
  lon=-46.50 → distância=0.1700
  lon=-46.25 → distância=0.0800

Ponto mais próximo: longitude -46.25 (índice 2)


In [ ]:
ponto_mais_proximo <- function(arquivo, santos_lat = -23.93, santos_lon = -46.33) {

  nc <- nc_open(arquivo)
  lat <- ncvar_get(nc, "latitude")
  lon <- ncvar_get(nc, "longitude")
  nc_close(nc)

  # Encontrar índices mais próximos
  idx_lat <- which.min(abs(lat - santos_lat))
  idx_lon <- which.min(abs(lon - santos_lon))

  cat(sprintf("Arquivo: %s\n", arquivo))
  cat(sprintf("  Ponto mais próximo → lat: %.4f (idx=%d) | lon: %.4f (idx=%d)\n",
              lat[idx_lat], idx_lat,
              lon[idx_lon], idx_lon))

  return(list(
    lat     = lat[idx_lat],
    lon     = lon[idx_lon],
    idx_lat = idx_lat,
    idx_lon = idx_lon
  ))
}

In [ ]:
arquivos <- list.files(pattern = "\\.nc$")
for (arq in arquivos) ponto_mais_proximo(arq)

Arquivo: variavel1.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.2500 (idx=2)
Arquivo: variavel2.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.2500 (idx=2)
Arquivo: variavel3.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)
Arquivo: variavel4.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.2500 (idx=2)
Arquivo: variavel5.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)
Arquivo: variavel6.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)
Arquivo: variavel7.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)
Arquivo: variavel8.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)
Arquivo: variavel9.nc
  Ponto mais próximo → lat: -24.0000 (idx=1) | lon: -46.5000 (idx=1)


In [ ]:
ler_nc <- function(arquivo) {
  nc <- nc_open(arquivo)

  # Definições dentro da função
  santos_lat   <- -23.93
  santos_lon   <- -46.33
  vars_ignorar <- c("number", "expver")

  # Coordenadas
  lat <- ncvar_get(nc, "latitude")
  lon <- ncvar_get(nc, "longitude")
  idx_lat <- which.min(abs(lat - santos_lat))
  idx_lon <- which.min(abs(lon - santos_lon))

  # Tempo
  tempo_raw <- ncvar_get(nc, "valid_time")
  datetime  <- as.POSIXct(tempo_raw, origin = "1970-01-01", tz = "UTC")

  # Variável principal
  nome_var  <- names(nc$var)[!names(nc$var) %in% vars_ignorar][1]
  dados_raw <- ncvar_get(nc, nome_var)

  # Extrair valores conforme dimensão do array
  if (is.matrix(dados_raw)) {
    # dimensão: 2 x 8760 — extrai a linha do ponto mais próximo
    valores <- dados_raw[idx_lon, ]
  } else {
    # dimensão: 8760 — vetor simples, usa direto
    valores <- as.numeric(dados_raw)
  }

  nc_close(nc)

  cat(sprintf("✓ %-6s | lat=%.2f lon=%.2f | %d obs\n",
              nome_var, lat[idx_lat], lon[idx_lon], length(valores)))

  tibble(
    datetime    = datetime,
    !!nome_var := as.numeric(valores)
  )
}

In [ ]:
arquivos <- list.files(pattern = "\\.nc$")

df_final <- arquivos %>%
  map(ler_nc) %>%
  reduce(full_join, by = "datetime") %>%
  arrange(datetime)

✓ u10    | lat=-24.00 lon=-46.25 | 8760 obs
✓ v10    | lat=-24.00 lon=-46.25 | 8760 obs
✓ pp1d   | lat=-24.00 lon=-46.50 | 8760 obs
✓ msl    | lat=-24.00 lon=-46.25 | 8760 obs
✓ mwd    | lat=-24.00 lon=-46.50 | 8760 obs
✓ cdww   | lat=-24.00 lon=-46.50 | 8760 obs
✓ wss    | lat=-24.00 lon=-46.50 | 8760 obs
✓ msqs   | lat=-24.00 lon=-46.50 | 8760 obs
✓ hmax   | lat=-24.00 lon=-46.50 | 8760 obs


In [ ]:
# ── Verificação ───────────────────────────────────────────────
cat("\nDimensões:", nrow(df_final), "x", ncol(df_final), "\n")
cat("Período:", format(min(df_final$datetime), "%Y-%m-%d %H:%M"),
    "a", format(max(df_final$datetime), "%Y-%m-%d %H:%M"), "\n")
cat("\nDados faltantes:\n")
print(colSums(is.na(df_final)))
cat("\nPrimeiras linhas:\n")
print(head(df_final))
glimpse(df_final)


Dimensões: 8760 x 10 
Período: 2025-01-01 00:00 a 2025-12-31 23:00 

Dados faltantes:
datetime      u10      v10     pp1d      msl      mwd     cdww      wss 
       0        0        0        0        0        0        0        0 
    msqs     hmax 
       0        0 

Primeiras linhas:
# A tibble: 6 × 10
  datetime              u10    v10  pp1d     msl   mwd     cdww    wss    msqs
  <dttm[1d]>          <dbl>  <dbl> <dbl>   <dbl> <dbl>    <dbl>  <dbl>   <dbl>
1 2025-01-01 00:00:00 -4.26 -0.587  9.93 101335.  150. 0.000841 0.0130 0.00343
2 2025-01-01 01:00:00 -4.15 -0.522  9.97 101353.  150. 0.000838 0.0130 0.00340
3 2025-01-01 02:00:00 -4.12 -0.621  9.99 101354.  149. 0.000815 0.0130 0.00337
4 2025-01-01 03:00:00 -3.68 -0.777 10.0  101296.  149. 0.000773 0.0131 0.00332
5 2025-01-01 04:00:00 -3.09 -0.552 10.0  101252.  149. 0.000724 0.0131 0.00325
6 2025-01-01 05:00:00 -2.45 -0.197 10.0  101234.  148. 0.000687 0.0130 0.00313
# ℹ 1 more variable: hmax <dbl>
Rows: 8,760
Columns: 10
$ d

In [ ]:
df_final

datetime,u10,v10,pp1d,msl,mwd,cdww,wss,msqs,hmax
<dttm[1d]>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025-01-01 00:00:00,-4.2586823,-0.587051392,9.932961,101335.4,150.3298,0.0008412045,0.01300998,0.003429498,2.337203
2025-01-01 01:00:00,-4.1511536,-0.522460938,9.966652,101352.9,149.8325,0.0008383510,0.01302727,0.003404713,2.352866
2025-01-01 02:00:00,-4.1249237,-0.621185303,9.993019,101354.5,149.3564,0.0008151513,0.01304291,0.003365618,2.365037
2025-01-01 03:00:00,-3.6797485,-0.777221680,10.013039,101295.8,148.8990,0.0007729949,0.01305439,0.003316987,2.373783
2025-01-01 04:00:00,-3.0907593,-0.552154541,10.027199,101251.7,148.5655,0.0007244621,0.01305060,0.003245468,2.378024
2025-01-01 05:00:00,-2.4466400,-0.196594238,10.036476,101234.1,148.4647,0.0006870215,0.01302527,0.003131984,2.374745
2025-01-01 06:00:00,-2.1089630,-0.338851929,10.041359,101198.0,148.6026,0.0006861491,0.01297547,0.002982760,2.364781
2025-01-01 07:00:00,-1.6513062,-0.528015137,10.041359,101182.9,148.9300,0.0006851763,0.01290855,0.002824101,2.349154
2025-01-01 08:00:00,-0.7988892,0.003189087,10.037453,101189.1,149.3990,0.0006843021,0.01282473,0.002667530,2.329506


In [ ]:
colnames(df_final) <- c("data_hora", "vento_u", "vento_v",
                         "periodo_onda_pico", "pressao_media_mar",
                         "direcao_media_onda", "coef_arrasto_onda",
                          "assimetria_espectral_onda", "inclinacao_media_quadratica_onda", "altura_maxima_onda")

In [ ]:
df_final

data_hora,vento_u,vento_v,periodo_onda_pico,pressao_media_mar,direcao_media_onda,coef_arrasto_onda,assimetria_espectral_onda,inclinacao_media_quadratica_onda,altura_maxima_onda
<dttm[1d]>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025-01-01 00:00:00,-4.2586823,-0.587051392,9.932961,101335.4,150.3298,0.0008412045,0.01300998,0.003429498,2.337203
2025-01-01 01:00:00,-4.1511536,-0.522460938,9.966652,101352.9,149.8325,0.0008383510,0.01302727,0.003404713,2.352866
2025-01-01 02:00:00,-4.1249237,-0.621185303,9.993019,101354.5,149.3564,0.0008151513,0.01304291,0.003365618,2.365037
2025-01-01 03:00:00,-3.6797485,-0.777221680,10.013039,101295.8,148.8990,0.0007729949,0.01305439,0.003316987,2.373783
2025-01-01 04:00:00,-3.0907593,-0.552154541,10.027199,101251.7,148.5655,0.0007244621,0.01305060,0.003245468,2.378024
2025-01-01 05:00:00,-2.4466400,-0.196594238,10.036476,101234.1,148.4647,0.0006870215,0.01302527,0.003131984,2.374745
2025-01-01 06:00:00,-2.1089630,-0.338851929,10.041359,101198.0,148.6026,0.0006861491,0.01297547,0.002982760,2.364781
2025-01-01 07:00:00,-1.6513062,-0.528015137,10.041359,101182.9,148.9300,0.0006851763,0.01290855,0.002824101,2.349154
2025-01-01 08:00:00,-0.7988892,0.003189087,10.037453,101189.1,149.3990,0.0006843021,0.01282473,0.002667530,2.329506


In [ ]:
write.csv(df_final, "df_final.csv", row.names = FALSE)